<a href="https://colab.research.google.com/github/samjurassic/datascience-demo/blob/main/coda/HBS_CoDA_Python_Part2_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HBS CoDA: NLP With Python

Learning objectives include:

• Understand vector embeddings of text, TF-IDF matrices

• Build a topic model on a corpus of text using BERTopic and UMAP

• Visualize topic model results

• Use a local LLM for classifying text data

In [ ]:
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [ ]:
!pip install --quiet bertopic datasets

## Constructing the TF-iDF Matrix

In [ ]:
# A tiny, low-dimensional dataset
corpus = [
    "the robot makes a car",     # Doc 0
    "the robot makes a pizza",   # Doc 1
    "the human eats a pizza"     # Doc 2
]

# 2. Extract the vocabulary (all unique words, sorted alphabetically)
vocab = sorted(set(word for doc in corpus for word in doc.split()))
print(f"Vocabulary ({len(vocab)} words): {vocab}\n")

# 3. Calculate TF (Term Frequency)
# Formula: (Count of word in doc) / (Total words in doc)
def compute_tf(corpus, vocab):
    tf_data = []
    for doc in corpus:
        words = doc.split()
        doc_length = len(words)

        # Count words and divide by total words in that document
        tf_row = {word: words.count(word) / doc_length for word in vocab}
        tf_data.append(tf_row)

    return pd.DataFrame(tf_data, index=["Doc 0", "Doc 1", "Doc 2"])

tf_matrix = compute_tf(corpus, vocab)
print("--- TERM FREQUENCY (TF) MATRIX ---")
display(tf_matrix) # 'display' looks really nice in Colab

In [ ]:
# Calculate IDF (Inverse Document Frequency)
# Formula: log(Total Documents / Number of Documents containing the word)
def compute_idf(corpus, vocab):
    N = len(corpus) # Total documents (3)
    idf_dict = {}

    for word in vocab:
        # How many documents contain this word?
        doc_count = sum(1 for doc in corpus if word in doc.split())

        # Calculate log (using base 10 for easier human reading)
        idf_dict[word] = math.log10(N / doc_count)

    return pd.DataFrame([idf_dict], index=["IDF Score"])

idf_matrix = compute_idf(corpus, vocab)
print("\n--- INVERSE DOCUMENT FREQUENCY (IDF) SCORES ---")
display(idf_matrix)

In [ ]:
# 5. Calculate final TF-IDF Matrix
# Formula: TF * IDF
def compute_tfidf(tf_matrix, idf_matrix):
    # Multiply the TF dataframe by the IDF dataframe row by row
    tfidf_matrix = tf_matrix.multiply(idf_matrix.iloc[0], axis=1)
    return tfidf_matrix

final_tfidf = compute_tfidf(tf_matrix, idf_matrix)
print("\n--- FINAL TF-IDF MATRIX ---")
display(final_tfidf)

In [ ]:
# Note: Sklearn uses slightly more complex math behind the scenes
# (L2 normalization and smoothed IDF) to prevent division-by-zero errors,
# but the core concept is identical!
vectorizer = TfidfVectorizer()
sklearn_matrix = vectorizer.fit_transform(corpus)

# Convert to a dataframe so we can see it
sklearn_df = pd.DataFrame(
    sklearn_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Doc 0", "Doc 1", "Doc 2"]
)

print("\n--- SCIKIT-LEARN TF-IDF MATRIX ---")
display(sklearn_df)

In [ ]:
# 1. Scikit-Learn's TF is just the raw count of words
tf_raw = []
for doc in corpus:
    words = doc.split()
    tf_raw.append([words.count(word) for word in vocab])
tf_raw = np.array(tf_raw)

# 2. Scikit-Learn's smoothed IDF formula: ln((1+N) / (1+df)) + 1
N = len(corpus)
idf_sklearn = []
for word in vocab:
    df = sum(1 for doc in corpus if word in doc.split())
    # Note the use of np.log (natural log) and the +1 smoothing
    idf_score = np.log((1 + N) / (1 + df)) + 1
    idf_sklearn.append(idf_score)
idf_sklearn = np.array(idf_sklearn)

# 3. Multiply TF * IDF
tfidf_raw = tf_raw * idf_sklearn

# 4. L2 Normalization (Scale each row so its Euclidean length is 1)
# Formula for a row: row / sqrt(sum(row^2))
row_norms = np.sqrt(np.sum(tfidf_raw**2, axis=1, keepdims=True))
tfidf_l2_normalized = tfidf_raw / row_norms

# Let's compare!
print("--- OUR MANUAL 'SKLEARN-STYLE' MATRIX ---")
display(pd.DataFrame(tfidf_l2_normalized, columns=vocab, index=["Doc 0", "Doc 1", "Doc 2"]).round(4))

print("\n--- ACTUAL SCIKIT-LEARN MATRIX ---")
display(sklearn_df.round(4))

In [ ]:
# By default, sklearn ignores single-character words.
# We can override the 'token_pattern' to include words of length 1 (\w+)
vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')

sklearn_matrix = vectorizer.fit_transform(corpus)

# Convert to a dataframe so we can see it
sklearn_df = pd.DataFrame(
    sklearn_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Doc 0", "Doc 1", "Doc 2"]
)

print("\n--- SCIKIT-LEARN TF-IDF MATRIX (With 'a' included) ---")
display(sklearn_df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Let's create 3 imaginary documents reduced to just 2 dimensions (X and Y)
# Think of X as "Pet-ness" and Y as "Finance-ness"
vec_A = np.array([[0.8, 0.1]])  # "I love my dog"
vec_B = np.array([[0.9, 0.2]])  # "Puppies are great" (Similar angle to A)
vec_C = np.array([[0.1, 0.9]])  # "Stock market crashed" (Completely different angle)

# Calculate Cosine Similarities against Vector A
sim_AB = cosine_similarity(vec_A, vec_B)[0][0]
sim_AC = cosine_similarity(vec_A, vec_C)[0][0]

# --- PLOTTING ---
plt.figure(figsize=(8, 6))
ax = plt.gca()

# Plot the vectors as arrows from the origin (0,0)
ax.quiver(0, 0, vec_A[0,0], vec_A[0,1], angles='xy', scale_units='xy', scale=1, color='blue', label=f'Doc A (Dog)')
ax.quiver(0, 0, vec_B[0,0], vec_B[0,1], angles='xy', scale_units='xy', scale=1, color='cyan', label=f'Doc B (Puppy) | Sim to A: {sim_AB:.2f}')
ax.quiver(0, 0, vec_C[0,0], vec_C[0,1], angles='xy', scale_units='xy', scale=1, color='red',  label=f'Doc C (Stocks) | Sim to A: {sim_AC:.2f}')

# Formatting the plot to look good in Colab
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.title("Cosine Similarity: Measuring the Angle Between Vectors", fontsize=14)
plt.xlabel("Dimension 1 (e.g., Pet-ness)")
plt.ylabel("Dimension 2 (e.g., Finance-ness)")
plt.legend(loc='upper left')
plt.show()

In [ ]:
import seaborn as sns
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# 1. Three sentences: Two are semantically similar (but share no nouns!), one is different.
sentences = [
    "The golden retriever chased the tennis ball.",
    "A happy dog ran after the round toy.",
    "Interest rates were raised by the federal reserve today."
]

# 2. Get the real 384-dimensional dense vectors
dense_vectors = embedding_model.encode(sentences)
print(f"Embeddings Shape: {dense_vectors.shape}") # Low, fixed dimensionality (e.g., 384)

# 3. Calculate the Cosine Similarity Matrix
# This compares every sentence against every other sentence
similarity_matrix = cosine_similarity(dense_vectors)

# 4. Plotting the Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(
    similarity_matrix,
    annot=True,          # Show the exact numbers
    cmap='Blues',        # Dark blue = highly similar
    vmin=0, vmax=1,      # Cosine sim for text is usually between 0 and 1
    xticklabels=["Doc 0 (Retriever)", "Doc 1 (Dog)", "Doc 2 (Finance)"],
    yticklabels=["Doc 0", "Doc 1", "Doc 2"]
)

plt.title("Real 384-D Text Embeddings Similarity Matrix", fontsize=14)
plt.xticks(rotation=15)
plt.show()

Embed text (done in Module 1).
Reduce dimensions (UMAP).
Cluster (HDBSCAN).
Extract topic words (c-TF-IDF).

In [ ]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN


# A small toy dataset to demonstrate clustering
docs = [
    # Topic 1: Baseball
    "The pitcher looked in for the sign before winding up for the throw.",
    "A loud crack echoed through the stadium as the bat connected with the ball.",
    "The umpire dusted off home plate and yelled for the batter to step in.",
    "Fans scrambled to catch the foul ball that landed in the upper deck.",
    "The game went into extra innings after the score remained tied in the ninth.",
    "He slid into second base just barely beating the tag from the shortstop.",
    "Managing the bullpen is a critical strategy for winning close games.",
    "The outfielder made a spectacular diving catch to save a run.",
    "Spring training allows players to warm up before the regular season begins.",
    "Hitting a grand slam is one of the most exciting moments in the sport.",

    # Topic 2: Steely Dan
    "Walter Becker and Donald Fagen formed the core of the band known for complex jazz structures.",
    "The album Aja is often cited as a masterpiece of audio engineering and production.",
    "Their lyrics often feature sarcastic humor and cryptic storytelling about eccentric characters.",
    "Steely Dan is famous for using a revolving door of top-tier session musicians.",
    "The blend of jazz harmonies with rock rhythms creates their signature sophisticated sound.",
    "Reelin' In the Years features one of the most recognizable guitar solos in classic rock.",
    "They were known for their obsessive perfectionism in the recording studio.",
    "After a long hiatus, the band returned to touring in the early nineties to great acclaim.",
    "The song Deacon Blues describes the fantasy of becoming a jazz saxophonist.",
    "Their music often defies easy categorization, bridging the gap between pop and jazz fusion.",

    # Topic 3: Seagulls
    "The seagull swooped down and snatched a french fry right out of my hand.",
    "Large flocks of white birds gathered along the shoreline during low tide.",
    "Their piercing cries can be heard echoing across the boardwalk early in the morning.",
    "Seagulls are incredibly opportunistic feeders and will eat almost anything they find.",
    "The grey and white feathers help them blend in with the cloudy coastal skies.",
    "Tourists are often warned not to feed the birds to avoid aggressive behavior.",
    "A solitary gull perched on the wooden piling, watching the fishing boats return.",
    "These coastal birds have adapted well to living in urban environments near the sea.",
    "They build their nests on high cliffs to protect their eggs from predators.",
    "The span of their wings allows them to glide effortlessly over the ocean currents."
]

df = pd.DataFrame({"text": docs})

# Look at the text
print(df['text'].head())
docs = df['text'].tolist()

# --- STEP 1: Setup Models (from previous step) ---
# We keep the small-data settings so it works on the 30 sentences
umap_model = UMAP(n_neighbors=3, n_components=2, min_dist=0.0, metric='cosine', random_state=1234)
hdbscan_model = HDBSCAN(min_cluster_size=3, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# --- STEP 2: Configure Stopword Removal ---
# We use CountVectorizer to remove standard English stopwords
vectorizer_model = CountVectorizer(stop_words="english")

# --- STEP 3: Run BERTopic ---
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model, # <--- Add the vectorizer here
    language="english"
)

topics, probs = topic_model.fit_transform(docs)

# --- STEP 4: Inspect the c-TF-IDF Matrix ---
# This matrix contains the scores for every word in every topic.
# Higher score = This word is more unique/important to this topic.

# Get the matrix (Topics x Words)
c_tf_idf_matrix = topic_model.c_tf_idf_

# Get the list of feature names (words)
feature_names = topic_model.vectorizer_model.get_feature_names_out()

# Convert to a Pandas DataFrame for easy viewing
df_tfidf = pd.DataFrame(
    c_tf_idf_matrix.toarray(),
    index=[f"Topic {i}" for i in topic_model.topic_labels_.keys()],
    columns=feature_names
)

# Show a subset of interesting words to prove it worked
# We pick words we know should be in our 3 topics
interesting_words = ["baseball", "pitcher", "guitar", "jazz", "bird", "seagull"]

# Filter the dataframe to show only these words (if they exist in the vocab)
# Note: We use an intersection check in case a word was dropped
valid_cols = [w for w in interesting_words if w in df_tfidf.columns]
print(df_tfidf[valid_cols].round(3))

In [ ]:
# Visualize the actual documents (dots) in vector space
# heavily reduces dimensionality to 2D so we can see it
fig = topic_model.visualize_documents(docs)

# This returns a Plotly figure (interactive)
fig.show()

In [ ]:
topic_model.visualize_topics()

In [ ]:
# Visualize the keywords
topic_model.visualize_barchart()